# 07 · Anomali Tespiti
Normal davranıştan sapan gözlemleri bulmak. Kullanım: kredi kartı dolandırıcılığı, ağ saldırısı, sensör arızası, üretim hataları.

| Yöntem | Fikir |
|---|---|
| **IQR / Z-skor** | İstatistiksel eşik (tek değişkenli) |
| **Isolation Forest** | Anomaliler rastgele bölmelerle *daha az adımda* izole edilir |
| **Local Outlier Factor (LOF)** | Komşularına göre yoğunluğu düşük olan noktalar anomalidir |

Veri: Gerçek anomalileri bildiğimiz sentetik 2D veri (değerlendirme yapabilmek için).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
%matplotlib inline

from sklearn.datasets import make_blobs
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import precision_score, recall_score, f1_score

In [ ]:
rng = np.random.default_rng(42)
X_normal, _ = make_blobs(n_samples=475, centers=[[0, 0], [5, 5]], cluster_std=1.0, random_state=42)
X_anomali = rng.uniform(-6, 11, size=(25, 2))
X = np.vstack([X_normal, X_anomali])
gercek = np.r_[np.zeros(475), np.ones(25)].astype(int)    # 1 = anomali

plt.figure(figsize=(6.5, 5.5))
plt.scatter(*X[gercek == 0].T, s=12, c="tab:blue", label="normal")
plt.scatter(*X[gercek == 1].T, s=40, c="tab:red", marker="x", label="gerçek anomali")
plt.legend(); plt.title("Veri (%5 anomali)"); plt.show()

**Yorum:** İki yoğun normal grup ve etrafa dağılmış 25 anomali var. Bazı rastgele anomaliler tesadüfen normal grupların içine düşmüş olabilir — onları bulmak her yöntem için zordur.

## 1. Basit yöntem: Z-skor

In [ ]:
z = np.abs((X - X.mean(0)) / X.std(0))
z_tahmin = (z > 2.5).any(axis=1).astype(int)
print(f"Z-skor → Precision={precision_score(gercek, z_tahmin):.2f}  Recall={recall_score(gercek, z_tahmin):.2f}")

**Yorum:** Z-skor her değişkene ayrı bakar ve tek merkezli dağılım varsayar; iki küme arasındaki boşluktaki anomalileri kaçırır.

## 2. Isolation Forest ve LOF

In [ ]:
iso = IsolationForest(contamination=0.05, random_state=42).fit(X)
iso_tahmin = (iso.predict(X) == -1).astype(int)          # -1 = anomali

lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05)
lof_tahmin = (lof.fit_predict(X) == -1).astype(int)

tablo = pd.DataFrame({ad: {"Precision": precision_score(gercek, t), "Recall": recall_score(gercek, t), "F1": f1_score(gercek, t)}
                      for ad, t in [("Z-skor", z_tahmin), ("Isolation Forest", iso_tahmin), ("LOF", lof_tahmin)]}).T
tablo.round(3)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5.5))
xx, yy = np.meshgrid(np.linspace(-7, 12, 200), np.linspace(-7, 12, 200))
skor = iso.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax[0].contourf(xx, yy, skor, levels=20, cmap="RdBu", alpha=0.6)
for a, t, ad in [(ax[0], iso_tahmin, "Isolation Forest"), (ax[1], lof_tahmin, "LOF")]:
    a.scatter(*X[t == 0].T, s=10, c="tab:blue"); a.scatter(*X[t == 1].T, s=50, c="tab:red", marker="x")
    a.set_title(f"{ad} – kırmızı = anomali dedi")
plt.tight_layout(); plt.show()

**Yorum:** Isolation Forest'ın karar yüzeyinde koyu kırmızı bölgeler anomali skorunun yüksek olduğu yerlerdir. Her iki yöntem de çok değişkenli yapıyı dikkate aldığı için Z-skordan belirgin şekilde daha iyi çalışır. `contamination` parametresi beklenen anomali oranıdır; gerçek hayatta bilinmez, iş bilgisiyle tahmin edilmelidir.

## Özet
### Data Analysis Key Findings
- Çok değişkenli yöntemler (Isolation Forest, LOF) tek değişkenli Z-skor yöntemine göre çok daha yüksek F1 sağladı.
- Normal grupların içine düşen anomaliler hiçbir yöntemle bulunamaz — bu, özellik mühendisliğinin önemini gösterir.
### Insights or Next Steps
- Etiketli anomali verisi varsa problemi dengesiz sınıflandırma olarak ele alıp (class_weight, SMOTE) gözetimli modeller deneyin.